# DS300 FacetRec — Chạy Full Demo trên Kaggle (Tải trực tiếp từ Hugging Face ~500MB/s — Chỉ cần 1 Dataset Adapter!)

Bản rút gọn tối đa thao tác: **KHÔNG cần tạo hay gắn Dataset Base Models (Part 1 / Part 2)**.
Tận dụng đường truyền tốc độ cao của máy chủ Kaggle (~500–600 MB/s với Xet) để tải thẳng 5 Base Models + BERT vào ổ SSD cục bộ `/tmp/local_models` trong ~2.5–3 phút (không tốn mạng nhà bạn!).

### Thiết lập trên Kaggle trước khi chạy (Chỉ 3 bước!)
1. **Notebook options → Accelerator:** Chọn **`GPU T4 x2`** (2 × 15GB = 30GB VRAM).
2. **Notebook options → Internet:** Bật **`ON`**.
3. **Add Input → Datasets:** Chỉ cần gắn **duy nhất 1 Dataset** `adapter_4_conference_recommendation` (chứa `Qwen4B`, `Qwen8B`, `Qwen14B`, `Llama8B`, `Gemma9B`).
4. **Add-ons → Secrets:** Thêm **`HF_TOKEN`** (nếu muốn chạy `Llama3.1-8B` và `Gemma2-9B`; nếu không có `HF_TOKEN` thì 3 model Qwen3 vẫn chạy bình thường).

### Phân bổ VRAM (4 LLM thường trú — Chuyển đổi 0 giây cho 4/5 model)
- **GPU 0 (15GB):** `Qwen3-4B` (~3.0GB) + `Qwen3-8B` (~5.6GB) + `BERT-base-uncased` (~0.5GB) ≈ **9.1 GB / 15 GB**
- **GPU 1 (15GB):** `Llama3.1-8B` (~5.6GB) + `Gemma2-9B` (~6.4GB) ≈ **12.0 GB / 15 GB**
- **`Qwen3-14B`** đã tải sẵn trên SSD `/tmp/local_models` → khi bấm nạp trong ngăn **Hệ thống**, backend tự giải phóng LRU trên GPU 1 và nạp trong ~15–20s.

In [ ]:
# ==============================================================================
# CELL 1: Cấu hình chung & Kiểm tra phần cứng Kaggle (Direct Hugging Face Mode)
# ==============================================================================
import os
import subprocess
from pathlib import Path

# Bật tải tốc độ cao Xet (~500-600 MB/s trên mạng server Kaggle)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# 1. Cấu hình Repo
REPO_URL = "https://github.com/Hakuchoiii2/Paper-and-conference-recommendation.git"
BRANCH = "feat/demo"
WORK_DIR = Path("/tmp/ds300_demo")
HF_CACHE_DIR = Path("/tmp/hf_cache")
LOCAL_SSD_MODELS = Path("/tmp/local_models")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SSD_MODELS.mkdir(parents=True, exist_ok=True)

# 2. Cấu hình Adapter (chỉ cần gắn duy nhất 1 Dataset adapter!)
ADAPTER_ROOT_OVERRIDE = "/kaggle/input/datasets/amalkin39/adapter-4-conference-recommendation/adapter"

# 3. Tải sẵn cả 5 model vào SSD /tmp (Cell 7 chỉ nạp 4 model PRELOAD_MODELS vào VRAM)
PRECACHE_ALL_5_MODELS = True

# 4. Danh sách 4 LLM nạp sẵn vào 2 GPU (Chuyển đổi 0 giây cho 4/5 model!)
# (Nếu chưa có HF_TOKEN, tự động bỏ qua Llama/Gemma và nạp Qwen3-4B, Qwen3-8B, Qwen3-14B)
PRELOAD_MODELS = ["Qwen3-4B", "Qwen3-8B", "Llama3.1-8B", "Gemma2-9B"]

GPU_ASSIGNMENTS = {
    "Qwen3-4B": 0,
    "Qwen3-8B": 0,
    "Llama3.1-8B": 1,
    "Gemma2-9B": 1,
    "Qwen3-14B": 1,
}

DEFAULT_MODEL = PRELOAD_MODELS[0] if PRELOAD_MODELS else "Qwen3-4B"

print("=== KIỂM TRA GPU ===")
subprocess.run(["nvidia-smi"], check=False)
print("\n=== KIỂM TRA DUNG LƯỢNG Ổ ĐĨA ===")
subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], check=False)

In [ ]:
# ==============================================================================
# CELL 2: Clone mã nguồn, Tìm Adapter & Đọc HF_TOKEN
# ==============================================================================
import os
import subprocess
from pathlib import Path

EXPECTED_ADAPTERS = ["Qwen4B", "Qwen8B", "Qwen14B", "Llama8B", "Gemma9B"]
HF_REPOS = {
    "Qwen3-4B": "Qwen/Qwen3-4B",
    "Qwen3-8B": "Qwen/Qwen3-8B",
    "Qwen3-14B": "Qwen/Qwen3-14B",
    "Llama3.1-8B": "meta-llama/Llama-3.1-8B-Instruct",
    "Gemma2-9B": "google/gemma-2-9b-it",
}
BERT_HF_REPO = "google-bert/bert-base-uncased"

if (WORK_DIR / ".git").exists():
    print(f"Repo đã tồn tại tại {WORK_DIR}, đang cập nhật nhánh {BRANCH}...")
    subprocess.run(["git", "-C", str(WORK_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORK_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORK_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Đang clone nhánh {BRANCH} vào {WORK_DIR}...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORK_DIR)],
        check=True,
    )

def find_adapter_root(override: str) -> Path:
    if override:
        p = Path(override)
        if p.exists():
            return p
        print(f"⚠️ Không thấy đường dẫn cố định {override}, chuyển sang tự động dò trong /kaggle/input...")

    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        raise FileNotFoundError("Không tìm thấy /kaggle/input. Bạn đã chạy trên Kaggle chưa?")

    best_dir = None
    best_score = 0
    for root, dirs, _ in os.walk(kaggle_input, followlinks=True):
        score = sum(1 for name in EXPECTED_ADAPTERS if name in dirs)
        if score > best_score:
            best_score = score
            best_dir = Path(root)
        if score == len(EXPECTED_ADAPTERS):
            break

    if best_dir is None or best_score == 0:
        raise FileNotFoundError(
            "Không tìm thấy thư mục adapter (Qwen4B/Qwen8B/Qwen14B/Llama8B/Gemma9B) trong /kaggle/input!"
        )
    return best_dir

ADAPTER_ROOT = find_adapter_root(ADAPTER_ROOT_OVERRIDE)
print(f"\n✅ Đã tìm thấy ADAPTER_ROOT: {ADAPTER_ROOT}")
for name in EXPECTED_ADAPTERS:
    sub = ADAPTER_ROOT / name
    has_cfg = (sub / "adapter_config.json").exists()
    status = "✅ Sẵn sàng" if has_cfg else "❌ Thiếu adapter_config.json"
    print(f"   - {name:8s}: {status} ({sub})")

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN").strip()
        print("\n✅ Đã đọc HF_TOKEN từ Kaggle Secrets (hỗ trợ đầy đủ 5/5 model).")
    except Exception:
        HF_TOKEN = ""
        print(
            "\n⚠️ Chưa có HF_TOKEN trong Kaggle Secrets -> tự động chạy 3 model Qwen3 (4B, 8B, 14B)."
        )
        PRELOAD_MODELS = ["Qwen3-4B", "Qwen3-8B", "Qwen3-14B"]

# Đường dẫn đích trên SSD /tmp (sẽ được tải ở Cell 6)
RESOLVED_BASE_MODELS = {k: str(LOCAL_SSD_MODELS / k) for k in HF_REPOS}
RESOLVED_BERT_MODEL = str(LOCAL_SSD_MODELS / "bert-base-uncased")

In [ ]:
# ==============================================================================
# CELL 3: Cài đặt thư viện Python & Tái tạo _local/demo_artifacts/
# ==============================================================================
import os
import subprocess
from pathlib import Path

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if Path(RESOLVED_BERT_MODEL).exists():
    os.environ["BERT_MODEL_PATH"] = RESOLVED_BERT_MODEL

packages = [
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.32.0",
    "pydantic>=2.9.0",
    "scikit-learn>=1.5.0",
    "python-dotenv>=1.0.0",
    "sentence-transformers>=3.0.0",
    "transformers>=4.46.0",
    "peft>=0.12.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "safetensors>=0.4.3",
    "sentencepiece>=0.2.0",
    "protobuf>=4.25.0",
    "pycloudflared",
    "requests",
]
print("Đang cài đặt thư viện Python (mất ~1-2 phút)...")
subprocess.run(["pip", "install", "-q", "-U", *packages], check=True)
print("✅ Cài đặt thư viện Python hoàn tất!")

artifacts_manifest = WORK_DIR / "_local" / "demo_artifacts" / "manifest.json"
if artifacts_manifest.exists():
    print(f"✅ Đã có sẵn demo_artifacts tại {artifacts_manifest.parent}")
else:
    print("Đang chạy build_demo_artifacts.py (nạp BERT-base-uncased và sinh vector 6 trường)...")
    subprocess.run(
        ["python", "demo/backend/scripts/build_demo_artifacts.py"],
        cwd=str(WORK_DIR),
        check=True,
    )
    print("✅ Tạo _local/demo_artifacts/ thành công!")

In [ ]:
# ==============================================================================
# CELL 4: Tạo demo/.env cho Kaggle Multi-GPU (4 LLM thường trú — LRU Pool tích hợp sẵn)
# ==============================================================================
from pathlib import Path

gpu_map_str = ",".join(f"{k}:{v}" for k, v in GPU_ASSIGNMENTS.items())

env_content = f"""# Auto-generated for Kaggle Multi-GPU runtime (4-model resident LRU pool)
ENABLE_REAL_LLM=true
DEVICE=cuda
LOAD_IN_4BIT=true
LOCAL_ACTIVE_MODEL={DEFAULT_MODEL}
KAGGLE_GPU_MAP={gpu_map_str}

HF_TOKEN={HF_TOKEN}
HF_HOME={HF_CACHE_DIR}
TRANSFORMERS_CACHE={HF_CACHE_DIR / 'hub'}

BASE_MODEL_QWEN4B={RESOLVED_BASE_MODELS['Qwen3-4B']}
BASE_MODEL_QWEN8B={RESOLVED_BASE_MODELS['Qwen3-8B']}
BASE_MODEL_QWEN14B={RESOLVED_BASE_MODELS['Qwen3-14B']}
BASE_MODEL_LLAMA8B={RESOLVED_BASE_MODELS['Llama3.1-8B']}
BASE_MODEL_GEMMA9B={RESOLVED_BASE_MODELS['Gemma2-9B']}

ADAPTER_ROOT={ADAPTER_ROOT}
ADAPTER_QWEN4B_PATH={ADAPTER_ROOT / 'Qwen4B'}
ADAPTER_QWEN8B_PATH={ADAPTER_ROOT / 'Qwen8B'}
ADAPTER_QWEN14B_PATH={ADAPTER_ROOT / 'Qwen14B'}
ADAPTER_LLAMA8B_PATH={ADAPTER_ROOT / 'Llama8B'}
ADAPTER_GEMMA9B_PATH={ADAPTER_ROOT / 'Gemma9B'}

EXPERIMENTS_ROOT=../paper-and-conference-recommendation-experiments
DEMO_ARTIFACTS_DIR=../_local/demo_artifacts
BERT_MODEL_PATH={RESOLVED_BERT_MODEL}

PAPER_CORPUS_PATH=../data/processed/papers.jsonl
CORPUS_FACETS_PATH=../data/exp_a/generated/facets_silver.jsonl
CORPUS_EVIDENCE_PATH=../data/exp_a/generated/annotation_metadata.jsonl

CONFERENCE_CATALOG_PATH=../paper-and-conference-recommendation-experiments/data/conference_recommendation_dataset/5_mapping_reviewed/splits/temporal_2015_2016_2019_clean/conference_catalog.jsonl
CONFERENCE_TRAIN_PATH=../paper-and-conference-recommendation-experiments/data/conference_recommendation_dataset/5_mapping_reviewed/splits/temporal_2015_2016_2019_clean/train.jsonl
CONFERENCE_TEST_PATH=../paper-and-conference-recommendation-experiments/data/conference_recommendation_dataset/5_mapping_reviewed/splits/temporal_2015_2016_2019_clean/test.jsonl
GENERATED_FACETS_DIR=../paper-and-conference-recommendation-experiments/data/conference_recommendation_dataset/generated_facets

PROMPT_GUIDELINE_PATH=../paper-and-conference-recommendation-experiments/data/exp_a/prompts/guideline_gold_v1.md
PROMPT_EXTRACTION_PATH=../paper-and-conference-recommendation-experiments/data/exp_a/prompts/extraction_gold_v1.md
"""

env_path = WORK_DIR / "demo" / ".env"
env_path.write_text(env_content, encoding="utf-8")
print(f"✅ Đã ghi cấu hình tại: {env_path}")

# Kiểm tra HF_TOKEN cho model gated (chỉ khóa nếu chưa có local Dataset và chưa có HF_TOKEN)
service_path = WORK_DIR / "demo" / "backend" / "service.py"
service_text = service_path.read_text(encoding="utf-8")
service_anchor = "        self.runtime = RuntimeStatus("
service_patch = """        for _m in self.models:
            if _m.supported and _m.model_key in ("Llama3.1-8B", "Gemma2-9B") and not os.getenv("HF_TOKEN"):
                _spec = MODEL_REGISTRY.get(_m.model_key)
                _resolved = self.model_runner._resolve_base_model_name(_spec) if _spec else ""
                if not (_resolved and Path(_resolved).exists()):
                    _m.supported = False
                    _m.unavailable_reason = "Cần cấu hình HF_TOKEN trong Kaggle Secrets (hoặc gắn Dataset trọng số local) cho model gated."

        self.runtime = RuntimeStatus("""
if "model gated" not in service_text and service_anchor in service_text:
    service_text = service_text.replace(service_anchor, service_patch, 1)
    service_path.write_text(service_text, encoding="utf-8")
    print("✅ Đã bổ sung kiểm tra HF_TOKEN (bỏ qua nếu đã có Kaggle Dataset local) trong service.py.")
else:
    print("✅ Backend đã có sẵn Multi-GPU LRU Pool & kiểm tra model tự động!")

In [ ]:
# ==============================================================================
# CELL 5: Đảm bảo Node.js >= 22 & Build Frontend React/Vite
# ==============================================================================
import os
import subprocess
from pathlib import Path

NODE_DIR = Path("/tmp/node-v22")
env_node = os.environ.copy()

def get_node_major(env):
    try:
        out = subprocess.check_output(["node", "-v"], env=env, text=True).strip()
        return int(out.lstrip("v").split(".")[0]), out
    except Exception:
        return 0, "none"

major, ver = get_node_major(env_node)
if major < 20:
    if not (NODE_DIR / "bin" / "node").exists():
        print(f"Node hiện tại ({ver}) quá cũ cho Vite 8. Đang tải Node.js 22 LTS vào {NODE_DIR}...")
        NODE_URL = "https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz"
        tar_path = Path("/tmp/node22.tar.xz")
        subprocess.run(["curl", "-fsSL", NODE_URL, "-o", str(tar_path)], check=True)
        NODE_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["tar", "-xJf", str(tar_path), "--strip-components=1", "-C", str(NODE_DIR)],
            check=True,
        )
        tar_path.unlink(missing_ok=True)
    env_node["PATH"] = f"{NODE_DIR / 'bin'}:{env_node.get('PATH', '')}"
    major, ver = get_node_major(env_node)

print(f"✅ Sử dụng Node.js: {ver}")

frontend_dir = WORK_DIR / "demo" / "frontend"
print("Đang cài npm packages & build giao diện tĩnh...")
subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=str(frontend_dir), env=env_node, check=True)
subprocess.run(["npm", "run", "build"], cwd=str(frontend_dir), env=env_node, check=True)

dist_index = frontend_dir / "dist" / "index.html"
assert dist_index.exists(), "Build frontend thất bại: không tìm thấy dist/index.html"
print(f"✅ Build Frontend thành công: {dist_index}")

In [ ]:
# ==============================================================================
# CELL 6: Tải trực tiếp Base Models + BERT từ Hugging Face vào SSD /tmp (~500MB/s)
# ==============================================================================
# - Dùng đường truyền nội bộ máy chủ Kaggle (không tốn mạng cá nhân).
# - Tự kiểm tra nếu đã tải đầy đủ trên /tmp/local_models thì bỏ qua ngay (0s).
import os
import shutil
import time
from pathlib import Path
from huggingface_hub import snapshot_download

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

ENV_KEY_MAP = {
    "Qwen3-4B": "BASE_MODEL_QWEN4B",
    "Qwen3-8B": "BASE_MODEL_QWEN8B",
    "Qwen3-14B": "BASE_MODEL_QWEN14B",
    "Llama3.1-8B": "BASE_MODEL_LLAMA8B",
    "Gemma2-9B": "BASE_MODEL_GEMMA9B",
}

ALL_MODEL_KEYS = ["Qwen3-4B", "Qwen3-8B", "Llama3.1-8B", "Gemma2-9B", "Qwen3-14B"]
MODELS_TO_PREPARE = ALL_MODEL_KEYS if PRECACHE_ALL_5_MODELS else list(PRELOAD_MODELS)

IGNORE_PATTERNS = [
    "*.msgpack", "*.h5", "*.ot", "*.onnx", "*.tflite",
    "tf_model*", "flax_model*", "rust_model*", "original/*", "pytorch_model*.bin",
]

def is_ready(dst: Path) -> bool:
    if not (dst / "config.json").exists():
        return False
    weights = list(dst.glob("*.safetensors")) + list(dst.glob("*.bin"))
    return len(weights) > 0

# 1. Tải BERT-base-uncased vào SSD /tmp
bert_dst = Path(RESOLVED_BERT_MODEL)
if is_ready(bert_dst):
    print(f"⚡ [BERT] Đã có sẵn trên SSD /tmp: {bert_dst} -> bỏ qua!")
else:
    print(f"📥 [BERT] Đang tải {BERT_HF_REPO} -> {bert_dst} ...")
    t0 = time.time()
    bert_dst.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=BERT_HF_REPO,
        local_dir=str(bert_dst),
        ignore_patterns=[p for p in IGNORE_PATTERNS if p != "pytorch_model*.bin"],
    )
    print(f"   ✅ Xong BERT trong {time.time() - t0:.1f}s!")

# 2. Tải các LLM từ Hugging Face vào SSD /tmp
for idx, key in enumerate(MODELS_TO_PREPARE, 1):
    repo_id = HF_REPOS[key]
    dst_local = LOCAL_SSD_MODELS / key

    if is_ready(dst_local):
        size_gb = sum(f.stat().st_size for f in dst_local.rglob("*") if f.is_file()) / (1024**3)
        print(f"\n⚡ [{idx}/{len(MODELS_TO_PREPARE)}] {key} đã có sẵn trên SSD /tmp ({size_gb:.2f} GB) -> bỏ qua!")
        RESOLVED_BASE_MODELS[key] = str(dst_local)
        continue

    if key in ("Llama3.1-8B", "Gemma2-9B") and not HF_TOKEN:
        print(f"\n⚠️ [{idx}/{len(MODELS_TO_PREPARE)}] Bỏ qua {key} vì chưa có HF_TOKEN.")
        RESOLVED_BASE_MODELS[key] = repo_id
        continue

    free_gb = shutil.disk_usage("/tmp").free / (1024**3)
    print(f"\n📥 [{idx}/{len(MODELS_TO_PREPARE)}] Đang tải {key} ({repo_id}) -> {dst_local} | /tmp trống: {free_gb:.1f} GB")
    if free_gb < 15:
        print("⚠️ Ổ /tmp còn dưới 15GB -> dừng tải thêm để tránh đầy đĩa!")
        break
    t0 = time.time()
    dst_local.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(dst_local),
        token=HF_TOKEN or None,
        ignore_patterns=IGNORE_PATTERNS,
    )
    RESOLVED_BASE_MODELS[key] = str(dst_local)
    size_gb = sum(f.stat().st_size for f in dst_local.rglob("*") if f.is_file()) / (1024**3)
    print(f"   ✅ Xong {key} ({size_gb:.2f} GB) trong {time.time() - t0:.1f}s!")

# 3. Cập nhật demo/.env trỏ thẳng vào SSD /tmp/local_models/...
env_path = WORK_DIR / "demo" / ".env"
if env_path.exists():
    env_lines = env_path.read_text(encoding="utf-8").splitlines()
    updated = []
    for ln in env_lines:
        replaced = False
        for mk, env_k in ENV_KEY_MAP.items():
            if ln.startswith(f"{env_k}="):
                updated.append(f"{env_k}={RESOLVED_BASE_MODELS[mk]}")
                replaced = True
                break
        if not replaced and ln.startswith("BERT_MODEL_PATH="):
            updated.append(f"BERT_MODEL_PATH={RESOLVED_BERT_MODEL}")
            replaced = True
        if not replaced:
            updated.append(ln)
    env_path.write_text("\n".join(updated) + "\n", encoding="utf-8")
    print("\n✅ Đã cập nhật demo/.env trỏ sang SSD /tmp/local_models!")

In [ ]:
# ==============================================================================
# CELL 7: Khởi chạy FastAPI + Nạp sẵn 3 LLM lên 2 GPU (theo dõi trực tiếp) + Mở Link Web
# ==============================================================================
import os
import subprocess
import time
import requests
from pycloudflared import try_cloudflare

LOG_FILE = Path("/tmp/uvicorn_demo.log")
backend_dir = WORK_DIR / "demo" / "backend"

# 1. Tắt server cũ nếu chạy lại cell
subprocess.run(["pkill", "-f", "uvicorn main:app"], check=False)
time.sleep(1)

# 2. Chạy FastAPI ngầm với PYTHONUNBUFFERED=1 để log đẩy ra tức thì
env_srv = os.environ.copy()
env_srv["PYTHONUNBUFFERED"] = "1"
env_srv["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env_srv["HF_HOME"] = str(HF_CACHE_DIR)
env_srv["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
if HF_TOKEN:
    env_srv["HF_TOKEN"] = HF_TOKEN

log_fp = open(LOG_FILE, "w", encoding="utf-8")
server_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(backend_dir),
    env=env_srv,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)

# 3. Đợi Backend sẵn sàng
print("⏳ Đang khởi động FastAPI Backend...")
ready = False
for _ in range(45):
    if server_proc.poll() is not None:
        break
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print("❌ Server không khởi động được! Log lỗi:")
    print(LOG_FILE.read_text(encoding="utf-8", errors="ignore")[-3000:])
    raise RuntimeError("FastAPI failed to start")

print("✅ FastAPI Backend đã sẵn sàng tại http://127.0.0.1:8000")

# Kiểm tra trạng thái 5 LLM
models = requests.get("http://127.0.0.1:8000/api/v1/models", timeout=5).json()
supported_keys = set()
print("\n📋 Trạng thái 5 LLM Extractor:")
for m in models:
    ok = m.get("supported", False)
    if ok:
        supported_keys.add(m["model_key"])
    note = m.get("unavailable_reason") or f"Sẵn sàng nạp (GPU {GPU_ASSIGNMENTS.get(m['model_key'], 0)})"
    print(f"   - {m['model_key']:12s} | supported={str(ok):5s} | {note}")

# 4. Nạp sẵn toàn bộ PRELOAD_MODELS vào 2 GPU và IN TRỰC TIẾP tiến độ từ log server
log_pos = LOG_FILE.stat().st_size if LOG_FILE.exists() else 0

def flush_new_logs():
    global log_pos
    if not LOG_FILE.exists():
        return
    with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
        f.seek(log_pos)
        chunk = f.read()
        log_pos = f.tell()
    for line in chunk.splitlines():
        if line.strip() and "GET /api/v1/jobs/" not in line and "GET /api/v1/runtime" not in line:
            print("   │", line, flush=True)

for idx, mk in enumerate(PRELOAD_MODELS, 1):
    if mk not in supported_keys:
        print(f"\n⚠️ Bỏ qua nạp sẵn {mk} (chưa supported).")
        continue
    print(f"\n🚀 [{idx}/{len(PRELOAD_MODELS)}] Đang nạp sẵn {mk} lên GPU {GPU_ASSIGNMENTS.get(mk, 0)}...", flush=True)
    t0 = time.time()
    res = requests.post(
        "http://127.0.0.1:8000/api/v1/runtime/model",
        json={"model_key": mk},
        timeout=10,
    )
    if res.status_code not in (200, 202):
        print(f"⚠️ Không thể gửi lệnh nạp {mk}: {res.text}")
        continue
    job_id = res.json().get("job_id")
    while True:
        flush_new_logs()
        job = requests.get(f"http://127.0.0.1:8000/api/v1/jobs/{job_id}", timeout=5).json()
        status = job.get("status")
        if status == "completed":
            flush_new_logs()
            print(f"✅ Xong {mk} trong {time.time() - t0:.1f}s!", flush=True)
            break
        if status == "failed":
            flush_new_logs()
            err = job.get("error") or {}
            print(f"❌ Nạp {mk} thất bại: {err.get('message') or err}", flush=True)
            break
        time.sleep(2)

print("\n=== VRAM ĐANG SỬ DỤNG TRÊN 2 GPU ===")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], check=False)

# 5. Mở Cloudflare Tunnel công khai
tunnel = try_cloudflare(port=8000)
print("\n" + "=" * 76)
print("🎉 DS300 FACETREC DEMO ĐANG CHẠY (3 LLM THƯỜNG TRÚ TRÊN 2 GPU — ĐỔI MODEL 0 GIÂY)!")
print(f"👉 LINK TRUY CẬP WEB: {tunnel.tunnel}")
print("=" * 76)
print("💡 Muốn xem tiến độ trực tiếp khi bấm 'Nạp model' hoặc 'Phân tích' trên web -> chạy Cell 8 bên dưới!")

In [ ]:
# ==============================================================================
# CELL 8 (Live Monitor): Theo dõi trực tiếp tiến độ nạp model, inference & VRAM
# ==============================================================================
# Chạy cell này để xem log trực tiếp mỗi khi bạn thao tác trên Web UI (bấm Stop Cell khi không cần xem nữa).
import subprocess
import time
from pathlib import Path

LOG_FILE = Path("/tmp/uvicorn_demo.log")
MONITOR_SECONDS = 7200  # Theo dõi & giữ phiên liên tục tối đa 2 tiếng

print("=== GPU VRAM & Ổ ĐĨA HIỆN TẠI ===")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv"], check=False)
subprocess.run(["df", "-h", "/tmp"], check=False)

print(f"\n📡 Đang theo dõi log server trực tiếp trong {MONITOR_SECONDS}s (bấm nút Stop ■ để dừng)...")
print("-" * 76)
if LOG_FILE.exists():
    # In trước 20 dòng gần nhất
    tail_lines = LOG_FILE.read_text(encoding="utf-8", errors="ignore").splitlines()[-20:]
    for ln in tail_lines:
        if "GET /api/v1/runtime" not in ln and "GET /api/v1/jobs/" not in ln:
            print(ln, flush=True)

    with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
        f.seek(0, 2)
        t_end = time.time() + MONITOR_SECONDS
        last_vram = time.time()
        while time.time() < t_end:
            line = f.readline()
            if line:
                if "GET /api/v1/runtime" not in line and "GET /api/v1/jobs/" not in line:
                    print(line.rstrip(), flush=True)
            else:
                if time.time() - last_vram >= 15:
                    out = subprocess.check_output(
                        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
                        text=True,
                    ).strip().replace("\n", " | ")
                    print(f"[VRAM] {out}", flush=True)
                    last_vram = time.time()
                time.sleep(0.5)
else:
    print("Chưa có file log.")